# Laboratorium: Przeszukiwanie przestrzeni stanów i heurystyki

To laboratorium dotyczy implementacji i porównania algorytmów:

- **BFS**
- **DFS**
- **A\***
- heurystyk dla problemu **8-puzzle**

## Cele

Po wykonaniu ćwiczenia student powinien umieć:

- modelować problem jako przestrzeń stanów,
- zaimplementować BFS, DFS i A*,
- porównać zachowanie algorytmów,
- zastosować heurystyki do 8-puzzle,
- zinterpretować różnice w liczbie odwiedzonych stanów i koszcie rozwiązania.

## Instrukcja pracy

Notebook zawiera:
1. krótkie wprowadzenie,
2. implementację algorytmów dla grafu,
3. implementację problemu 8-puzzle,
4. sekcję eksperymentalną,
5. pytania do interpretacji wyników.

Możesz:
- uruchamiać komórki krok po kroku,
- samodzielnie modyfikować dane wejściowe,
- uzupełniać odpowiedzi w komórkach Markdown.

## Część I — reprezentacja grafu i wspólne narzędzia

In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import heapq
import matplotlib.pyplot as plt
from typing import Callable

In [ ]:
Graph = dict[str, list[tuple[str, int]]]


@dataclass
class SearchResult:
    path: list
    visited_count: int
    cost: int | None = None


def reconstruct_path(parent: dict, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent.get(current)

    path.reverse()
    return path

### Przykładowy graf

Wagi krawędzi interpretujemy jako koszt przejścia.

In [ ]:
EXAMPLE_GRAPH: Graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("D", 5), ("E", 1)],
    "C": [("F", 1)],
    "D": [("G", 1)],
    "E": [("H", 1)],
    "F": [("G", 3)],
    "H": [("G", 1)],
    "G": [],
}

GRAPH_HEURISTIC = {
    "A": 4,
    "B": 3,
    "C": 3,
    "D": 1,
    "E": 1,
    "F": 2,
    "H": 1,
    "G": 0,
}


def graph_heuristic(node: str) -> int:
    return GRAPH_HEURISTIC[node]

### Wizualizacja grafu

Poniższa komórka rysuje dowolny skierowany graf zgodny ze strukturą `Graph = dict[str, list[tuple[str, int]]]` wraz z wagami krawędzi i opcjonalnymi wartościami heurystyki `h(n)`.
Możesz wyróżniać pojedyncze węzły listami `highlight_main` i `highlight_secondary`, a także podświetlić całą ścieżkę parametrem `highlight_path`.

In [ ]:
def draw_graph(
    graph: Graph,
    heuristic: dict[str, int] | None = None,
    highlight_main: list[str] | None = None,
    highlight_secondary: list[str] | None = None,
    highlight_path: list[str] | None = None,
    highlight_path_nodes: bool = False,
    title: str = "Graph",
    scale: float = 1.0,
    transparent_background: bool = False,
    show_edge_weights: bool = True,
) -> None:
    nodes = sorted(set(graph) | {neighbor for neighbors in graph.values() for neighbor, _ in neighbors})

    colours = {
        "std": {"face": "#dbeafe", "edge": "#1d4ed8"},
        "main": {"face": "#8afd95", "edge": "#01c021"},
        "secondary": {"face": "#fde68a", "edge": "#b45309"},
        "path": {"face": "#fca5a5", "edge": "#dc2626"},
    }

    if not nodes:
        raise ValueError("Graph is empty")

    indegree = {node: 0 for node in nodes}
    for neighbors in graph.values():
        for neighbor, _ in neighbors:
            indegree[neighbor] += 1

    roots = sorted(node for node in nodes if indegree[node] == 0)
    if not roots:
        roots = [nodes[0]]

    levels: dict[str, int] = {node: 0 for node in roots}
    queue = deque(roots)
    while queue:
        node = queue.popleft()
        for neighbor, _ in graph.get(node, []):
            candidate_level = levels[node] + 1
            if candidate_level > levels.get(neighbor, -1):
                levels[neighbor] = candidate_level
                queue.append(neighbor)

    for node in nodes:
        levels.setdefault(node, 0)

    level_groups: dict[int, list[str]] = {}
    for node in nodes:
        level_groups.setdefault(levels[node], []).append(node)

    positions: dict[str, tuple[float, float]] = {}
    x_spacing = 2.4
    y_spacing = 1.5
    for level in sorted(level_groups):
        group = sorted(level_groups[level])
        offset = (len(group) - 1) / 2
        for index, node in enumerate(group):
            positions[node] = (level * x_spacing, (offset - index) * y_spacing)

    for label, highlighted_nodes in {
        "highlight_main": highlight_main,
        "highlight_secondary": highlight_secondary,
        "highlight_path": highlight_path,
    }.items():
        if highlighted_nodes is None:
            continue
        for node in highlighted_nodes:
            if node not in positions:
                raise ValueError(f"Unknown node in {label}: {node}")

    path_edges = set()
    if highlight_path is not None:
        path_edges = set(zip(highlight_path, highlight_path[1:]))
        for source, target in path_edges:
            if target not in {neighbor for neighbor, _ in graph.get(source, [])}:
                raise ValueError(f"Path edge does not exist in graph: {source} -> {target}")

    fig_width = max(8, len(level_groups) * 2.2 * scale)
    fig_height = max(4.5, max(len(group) for group in level_groups.values()) * 1.6 * scale)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    if transparent_background:
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")

    for source, neighbors in graph.items():
        x1, y1 = positions[source]
        for target, weight in neighbors:
            x2, y2 = positions[target]
            dx = x2 - x1
            dy = y2 - y1
            is_path_edge = (source, target) in path_edges
            edge_colour = colours["path"]["edge"] if is_path_edge else "#64748b"
            edge_width = 2.8 if is_path_edge else 1.8
            ax.annotate(
                "",
                xy=(x2, y2),
                xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=edge_colour, lw=edge_width, shrinkA=28, shrinkB=28),
                zorder=1,
            )
            if show_edge_weights:
                ax.text(
                    x1 + dx * 0.5,
                    y1 + dy * 0.5 + 0.14,
                    str(weight),
                    fontsize=10,
                    ha="center",
                    va="center",
                    bbox=dict(
                        boxstyle="round,pad=0.2",
                        fc="#fee2e2" if is_path_edge else "white",
                        ec=colours["path"]["edge"] if is_path_edge else "none",
                        alpha=0.95,
                    ),
                    zorder=3,
                )

    for node, (x, y) in positions.items():
        if highlight_path is not None and highlight_path_nodes and node in highlight_path:
            is_highlighted = True
            colour_key = "path"
        elif highlight_main is not None and node in highlight_main:
            is_highlighted = True
            colour_key = "main"
        elif highlight_secondary is not None and node in highlight_secondary:
            is_highlighted = True
            colour_key = "secondary"
        else:
            is_highlighted = False
            colour_key = "std"

        line_width = 3 if is_highlighted else 2
        node_size = 2200 if is_highlighted else 1800
        ax.scatter(
            x,
            y,
            s=node_size * scale,
            color=colours[colour_key]["face"],
            edgecolors=colours[colour_key]["edge"],
            linewidths=line_width * scale,
            zorder=2,
        )

        label = node
        if heuristic is not None and node in heuristic:
            label = f"{node}\nh={heuristic[node]}"
        ax.text(x, y, label, ha="center", va="center", fontsize=11 * scale, weight="bold", zorder=4)

    xs = [x for x, _ in positions.values()]
    ys = [y for _, y in positions.values()]
    ax.set_title(title, fontsize=14 * scale, weight="bold")
    ax.set_xlim(min(xs) - 1.1, max(xs) + 1.1)
    ax.set_ylim(min(ys) - 1.1, max(ys) + 1.1)
    ax.axis("off")
    plt.show()


In [ ]:
draw_graph(
    EXAMPLE_GRAPH,
    GRAPH_HEURISTIC,
    highlight_main=["A"],
    highlight_secondary=["G"],
    highlight_path=["A", "B", "D", "G"],
    title="Example Graph",
    # highlight_path_nodes=True,
    # show_edge_weights=False,
    # transparent_background=True,
    scale=1.0,
)

## Część II — BFS

In [ ]:
def bfs(graph: Graph, start: str, goal: str, draw_graphs: bool = False) -> SearchResult:
    queue = deque([start])
    visited = {start}
    parent = {start: None}
    visited_count = 0

    while queue:
        node = queue.popleft()
        visited_count += 1

        if node == goal:
            path = reconstruct_path(parent, goal)
            return SearchResult(path=path, visited_count=visited_count, cost=len(path) - 1)

        for neighbor, _ in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                queue.append(neighbor)
        
        if draw_graphs:
            draw_graph(graph, highlight_main=[node], highlight_secondary=list(queue), title=f"BFS: Visited {visited_count} nodes", show_edge_weights=False)

    return SearchResult(path=[], visited_count=visited_count, cost=None)

In [ ]:
bfs_result = bfs(EXAMPLE_GRAPH, "A", "G", draw_graphs=False)
bfs_result

In [ ]:
draw_graph(EXAMPLE_GRAPH, highlight_main=["A", "G"], highlight_path=bfs_result.path, title="BFS Result", show_edge_weights=False)

### Pytania
1. Czy BFS znalazł ścieżkę o najmniejszej liczbie kroków?
2. Czy BFS uwzględnia wagi krawędzi?
3. Jak zmienia się złożoność pamięciowa BFS przy dużym współczynniku rozgałęzienia?

## Część III — DFS

In [ ]:
def dfs(graph: Graph, start: str, goal: str, draw_graphs: bool = False) -> SearchResult:
    # TODO Zaimplementuj algorytm DFS, analogicznie do BFS, ale z użyciem stosu zamiast kolejki.
    return SearchResult(path=[], visited_count=0, cost=None)

In [ ]:
dfs_result = dfs(EXAMPLE_GRAPH, "A", "G", draw_graphs=False)
dfs_result

In [ ]:
draw_graph(EXAMPLE_GRAPH, highlight_main=["A", "G"], highlight_path=dfs_result.path, title="DFS Result", show_edge_weights=False)

### Pytania
1. Czy DFS zwrócił tę samą ścieżkę co BFS?
2. Czy DFS gwarantuje rozwiązanie optymalne?
3. Dlaczego DFS zużywa zwykle mniej pamięci niż BFS?

## Część IV — Dijkstra

In [ ]:
def dijkstra(graph: Graph, start: str, goal: str, draw_graphs: bool = False) -> SearchResult:
    # TODO Zaimplementuj algorytm Dijkstry, używając kolejki priorytetowej (heapq) i śledząc koszty dojścia do poszczególnych wierzchołków.
    return SearchResult(path=[], visited_count=0, cost=None)

In [ ]:
dijkstra_result = dijkstra(EXAMPLE_GRAPH, "A", "G", draw_graphs=False)
dijkstra_result

In [ ]:
draw_graph(EXAMPLE_GRAPH, highlight_main=["A", "G"], highlight_path=dijkstra_result.path, title="Dijkstra Result")

### Pytania
1. Czym różni się wynik Dijkstry od BFS?
2. W jakich problemach BFS nie wystarcza, a Dijkstra jest poprawnym wyborem?

## Część V — A*

In [ ]:
def astar_graph(
    graph: Graph,
    start: str,
    goal: str,
    heuristic: Callable[[str], int],
    draw_graphs: bool = False
) -> SearchResult:
    # TODO Zaimplementuj algorytm A*, analogicznie do Dijkstry, ale z uwzględnieniem funkcji heurystycznej przy wyborze kolejnego wierzchołka do odwiedzenia.
    return SearchResult(path=[], visited_count=0, cost=None)

In [ ]:
astar_result = astar_graph(EXAMPLE_GRAPH, "A", "G", graph_heuristic, draw_graphs=False)
astar_result

In [ ]:
draw_graph(EXAMPLE_GRAPH, GRAPH_HEURISTIC, highlight_main=["A", "G"], highlight_path=astar_result.path, title="A* Result")

### Pytania
1. Jaką rolę odgrywa heurystyka w A*?
2. Co oznaczają składniki funkcji `f(n) = g(n) + h(n)`?
3. Czy A* zwrócił ten sam koszt co Dijkstra?

## Część VI — porównanie algorytmów dla grafu

In [ ]:
def compare_graph_algorithms():
    results = {
        "BFS": bfs(EXAMPLE_GRAPH, "A", "G"),
        "DFS": dfs(EXAMPLE_GRAPH, "A", "G"),
        "Dijkstra": dijkstra(EXAMPLE_GRAPH, "A", "G"),
        "A*": astar_graph(EXAMPLE_GRAPH, "A", "G", graph_heuristic),
    }

    for name, result in results.items():
        print(f"=== {name} ===")
        print("Ścieżka:", result.path)
        print("Koszt:", result.cost)
        print("Liczba odwiedzonych węzłów:", result.visited_count)
        print()

compare_graph_algorithms()

### Zadanie
Zmień strukturę grafu albo wagi krawędzi i sprawdź:

- czy BFS nadal daje najlepszy wynik,
- czy Dijkstra i A* zwracają taki sam koszt,
- czy liczba odwiedzonych węzłów przez A* zależy od heurystyki.

## Część VII — 8-puzzle

Reprezentacja stanu:
- liczby `1..8` oznaczają kafelki,
- `0` oznacza puste pole.

Stan zapisujemy jako krotkę 9-elementową, np.:

`(1, 2, 3, 4, 5, 6, 7, 8, 0)`

In [ ]:
PuzzleState = tuple[int, ...]
GOAL_STATE: PuzzleState = (1, 2, 3, 4, 5, 6, 7, 8, 0)


def print_puzzle(
    state: PuzzleState,
    *,
    highlighted_tiles: list[int] | None = None,
    title: str | None = None,
    ax: plt.Axes | None = None,
) -> None:
    created_figure = ax is None
    if created_figure:
        _, ax = plt.subplots(figsize=(3.6, 3.6))

    assert ax is not None
    ax.clear()
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")

    highlighted_set = set(highlighted_tiles or [])
    board_color = "#f5f5f5"
    border_color = "#333333"

    ax.add_patch(
        plt.Rectangle(
            (0, 0),
            3,
            3,
            facecolor=board_color,
            edgecolor=border_color,
            linewidth=2,
        )
    )

    for idx, value in enumerate(state):
        row, col = divmod(idx, 3)
        x, y = col, 2 - row

        is_highlighted = value in highlighted_set

        if value == 0:
            facecolor = "#f0f0f0"
            label = ""
        else:
            facecolor = "#ffe082" if is_highlighted else "#ffffff"
            label = str(value)

        tile_border_color = "#d97706" if is_highlighted else border_color
        tile_linewidth = 3 if is_highlighted else 2

        ax.add_patch(
            plt.Rectangle(
                (x + 0.05, y + 0.05),
                0.9,
                0.9,
                facecolor=facecolor,
                edgecolor=tile_border_color,
                linewidth=tile_linewidth,
            )
        )

        if label:
            ax.text(
                x + 0.5,
                y + 0.5,
                label,
                ha="center",
                va="center",
                fontsize=20,
                fontweight="bold",
                color="#1f2937",
            )

    if title is not None:
        ax.set_title(title)

    if created_figure:
        plt.show()

In [ ]:
def get_neighbors_puzzle(state: PuzzleState) -> list[PuzzleState]:
    neighbors = []
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)

    moves = []
    if row > 0:
        moves.append((-1, 0))
    if row < 2:
        moves.append((1, 0))
    if col > 0:
        moves.append((0, -1))
    if col < 2:
        moves.append((0, 1))

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc
        new_idx = new_row * 3 + new_col

        state_list = list(state)
        state_list[zero_idx], state_list[new_idx] = state_list[new_idx], state_list[zero_idx]
        neighbors.append(tuple(state_list))

    return neighbors

In [ ]:
start_puzzle = (1, 2, 3, 4, 5, 6, 0, 7, 8)

print("Stan początkowy:")
print_puzzle(start_puzzle, highlighted_tiles=[0], title="Stan początkowy")

print("Sąsiedzi:")
for neighbor in get_neighbors_puzzle(start_puzzle):
    print_puzzle(neighbor, highlighted_tiles=[0], title="Sąsiedzi")

## Część VIII — BFS dla 8-puzzle

In [ ]:
def bfs_puzzle(start: PuzzleState, goal: PuzzleState = GOAL_STATE) -> SearchResult:
    queue = deque([start])
    visited = {start}
    parent = {start: None}
    visited_count = 0

    while queue:
        state = queue.popleft()
        visited_count += 1

        if state == goal:
            path = reconstruct_path(parent, goal)
            return SearchResult(path=path, visited_count=visited_count, cost=len(path) - 1)

        for neighbor in get_neighbors_puzzle(state):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = state
                queue.append(neighbor)

    return SearchResult(path=[], visited_count=visited_count, cost=None)

In [ ]:
bfs_puzzle_result = bfs_puzzle(start_puzzle)
bfs_puzzle_result

## Część IX — heurystyki dla 8-puzzle

In [ ]:
def misplaced_tiles(state: PuzzleState, goal: PuzzleState = GOAL_STATE) -> int:
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != goal[i])


def manhattan_distance(state: PuzzleState, goal: PuzzleState = GOAL_STATE) -> int:
    distance = 0

    for tile in range(1, 9):
        current_index = state.index(tile)
        goal_index = goal.index(tile)

        current_row, current_col = divmod(current_index, 3)
        goal_row, goal_col = divmod(goal_index, 3)

        distance += abs(current_row - goal_row) + abs(current_col - goal_col)

    return distance

In [ ]:
print("Misplaced tiles:", misplaced_tiles(start_puzzle))
print("Manhattan distance:", manhattan_distance(start_puzzle))

## Część X — A* dla 8-puzzle

In [ ]:
def astar_puzzle(
    start: PuzzleState,
    goal: PuzzleState = GOAL_STATE,
    heuristic: Callable[[PuzzleState], int] = manhattan_distance,
) -> SearchResult:
    # TODO Zaimplementuj algorytm A* dla układanki 8-puzzle, używając funkcji heurystycznej do oceny kosztu dojścia do celu.
    return SearchResult(path=[], visited_count=0, cost=None)

In [ ]:
astar_misplaced_result = astar_puzzle(start_puzzle, heuristic=misplaced_tiles)
astar_manhattan_result = astar_puzzle(start_puzzle, heuristic=manhattan_distance)

print("A* + misplaced tiles:", astar_misplaced_result)
print("A* + Manhattan:", astar_manhattan_result)

## Część XI — wizualizacja rozwiązania

In [ ]:
def print_puzzle_solution(result: SearchResult) -> None:
    print(f"Liczba kroków: {result.cost}")
    print(f"Liczba odwiedzonych stanów: {result.visited_count}")
    print()

    for step_no, state in enumerate(result.path):
        print(f"Krok {step_no}:")
        print_puzzle(
            state,
            title=f"Krok {step_no}",
            highlighted_tiles=list(range(1, 9)) if state == GOAL_STATE else None,
        )

In [ ]:
print("=== Rozwiązanie BFS ===")
print_puzzle_solution(bfs_puzzle_result)

print("=== Rozwiązanie A* (Manhattan) ===")
print_puzzle_solution(astar_manhattan_result)

## Część XII — eksperymenty

Zmodyfikuj stan początkowy i porównaj zachowanie BFS oraz A*.

Przykładowe stany:

- `(1, 2, 3, 5, 0, 6, 4, 7, 8)`
- `(1, 3, 6, 5, 0, 2, 4, 7, 8)`
- `(7, 2, 4, 5, 0, 6, 8, 3, 1)` — trudny przypadek

In [ ]:
test_puzzle = (1, 2, 3, 5, 0, 6, 4, 7, 8)

print("Stan testowy:")
print_puzzle(test_puzzle)

result_bfs = bfs_puzzle(test_puzzle)
result_astar_misplaced = astar_puzzle(test_puzzle, heuristic=misplaced_tiles)
result_astar_manhattan = astar_puzzle(test_puzzle, heuristic=manhattan_distance)

print("BFS:", result_bfs)
print("A* misplaced:", result_astar_misplaced)
print("A* Manhattan:", result_astar_manhattan)

### Zadanie do wykonania
Dla co najmniej dwóch stanów początkowych porównaj:

- koszt rozwiązania,
- liczbę odwiedzonych stanów,
- różnicę między heurystyką `misplaced_tiles` i `manhattan_distance`.

Zapisz krótkie wnioski poniżej.

## Wnioski studenta

_Uzupełnij tę sekcję własnymi obserwacjami._

Przykładowe pytania pomocnicze:

1. Który algorytm był najefektywniejszy dla 8-puzzle?
2. Dlaczego A* zwykle odwiedza mniej stanów niż BFS?
3. Która heurystyka okazała się lepsza i dlaczego?
4. W jakich problemach DFS może być mimo wszystko użyteczny?

## Zadanie rozszerzające

1. Zaimplementuj sprawdzanie, czy stan 8-puzzle jest rozwiązywalny.
2. Dodaj pomiar czasu działania algorytmów.
3. Przygotuj tabelę porównującą BFS i A* dla kilku różnych stanów początkowych.
4. Spróbuj zaprojektować własną heurystykę dla 8-puzzle.